# D3-03 Prospective GWP100 across years and scenarios

This notebook uses the `edges` prospective climate-change method to reevaluate CH4 and N2O characterization factors across years and **all MESSAGE scenarios**, while keeping one tomato inventory fixed.

## Learning goals

After this notebook, you should be able to:

- load the packaged `Prospective_GWP100.json` method from `edges`
- inspect how `edges` stores scenario-specific CH4 and N2O CF tables by year
- map one tomato inventory once and iterate through scenario-specific CFs with repeated `evaluate_cfs()` calls
- compare how the same tomato activity gets different GWP100 scores across years and the full MESSAGE scenario family

## Background references

This notebook is based on the `edges` packaged method `Prospective_GWP100.json`, which cites:

- Watanabe, M. D. B. and Cherubini, F. (2026), *Prospective Characterization Factors for Assessing Climate Change Impacts in Life Cycle Assessments*, *Environmental Science & Technology*
- IPCC (2021), *Climate Change 2021: The Physical Science Basis*

The method keeps CO2 at a fixed CF of 1.0 and parameterises CH4 and N2O by IAM scenario and year.


## 1) Select one tomato dataset with CO2, CH4, and N2O emissions


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

import bw2data as bd
import edges
from edges import EdgeLCIA

pd.options.display.float_format = '{:,.4f}'.format


### Reuse the Swiss early-harvest greenhouse tomatoes from Day 2


In [ ]:
bd.projects.set_current('barcelona-rlcia-2026')
db = bd.Database('bafu')

TOMATO_NAME = 'Tomatoes conventional, hors-sol heated, at greenhouse, early harvest'
TOMATO_LOCATION = 'CH'

tomato_activity = next(
    ds for ds in db
    if ds['name'] == TOMATO_NAME and ds['location'] == TOMATO_LOCATION
)

pd.DataFrame([
    {
        'name': tomato_activity['name'],
        'reference product': tomato_activity.get('reference product'),
        'location': tomato_activity.get('location'),
        'unit': tomato_activity.get('unit'),
    }
])


### Inspect the direct climate-relevant biosphere flows


In [ ]:
target_flows = {'Carbon dioxide, fossil', 'Methane, fossil', 'Dinitrogen monoxide'}

flow_rows = []
for exc in tomato_activity.biosphere():
    if exc.input['name'] not in target_flows:
        continue
    flow_rows.append(
        {
            'supplier name': exc.input['name'],
            'amount': exc['amount'],
            'unit': exc.input['unit'],
        }
    )

pd.DataFrame(flow_rows).sort_values('supplier name').reset_index(drop=True)


## 2) Inspect the prospective GWP100 method shipped with `edges`


### Load the JSON file from the installed package


In [ ]:
prospective_method_path = Path(edges.__file__).resolve().parent / 'data' / 'Prospective_GWP100.json'
prospective_method = json.loads(prospective_method_path.read_text())

first_scenario = next(iter(prospective_method['scenarios']))
first_years = sorted(int(year) for year in prospective_method['scenarios'][first_scenario]['CF_CH4'])

pd.DataFrame([
    {
        'method': prospective_method['name'],
        'path': prospective_method_path.as_posix(),
        'unit': prospective_method['unit'],
        'number of exchanges': len(prospective_method['exchanges']),
        'number of scenarios': len(prospective_method['scenarios']),
        'year range': f'{min(first_years)}-{max(first_years)}',
    }
])


### Check how CO2, CH4, and N2O are encoded in the method


In [ ]:
pd.DataFrame([
    {
        'supplier name': exc['supplier']['name'],
        'categories': ' / '.join(exc['supplier'].get('categories', [])),
        'value': exc['value'],
    }
    for exc in prospective_method['exchanges']
    if exc['supplier']['name'] in target_flows and exc['supplier'].get('categories') == ['air']
])


### Keep four representative MESSAGE scenarios and evaluate every 20 years

The packaged method contains many IAM scenarios and annual values. To keep runtime reasonable in class, we:

- keep **four representative MESSAGE scenarios** spanning the forcing families,
- and evaluate them only every **20 years** from 2020 to 2100.

This still shows the scenario logic clearly while keeping the notebook fast enough for classroom use.


In [ ]:
def get_model_name(scenario):
    return scenario.split('_-_')[0] if '_-_' in scenario else scenario.split('-', 1)[0]


def get_pathway_label(scenario):
    if '_-_' in scenario:
        return scenario.split('_-_', 1)[1]
    return scenario.split('-', 1)[1] if '-' in scenario else scenario


def get_pathway_family(scenario):
    pathway = get_pathway_label(scenario)
    if 'rollBack' in pathway or pathway.endswith('-H') or '_H' in pathway:
        return 'high forcing'
    if 'NPi' in pathway or pathway.endswith('-M') or '_M' in pathway:
        return 'medium forcing'
    if 'NDC' in pathway or 'PkBudg1000' in pathway or 'LO' in pathway:
        return 'low forcing'
    if 'PkBudg650' in pathway or 'VL' in pathway:
        return 'very low forcing'
    if pathway.endswith('-L') or '_L' in pathway:
        return 'low forcing'
    return 'other'


In [ ]:
available_message_scenarios = {
    scenario
    for scenario in prospective_method['scenarios']
    if scenario.startswith('MESSAGE')
}

selected_scenarios = [
    'MESSAGE_-_SSP1-VL',  # very low forcing
    'MESSAGE_-_SSP1-L',   # low forcing
    'MESSAGE_-_SSP2-M',   # medium forcing
    'MESSAGE_-_SSP5-H',   # high forcing
]

missing_scenarios = [
    scenario for scenario in selected_scenarios if scenario not in available_message_scenarios
]
if missing_scenarios:
    raise ValueError(f'Missing MESSAGE scenarios in the prospective method: {missing_scenarios}')

scenario_metadata = pd.DataFrame([
    {
        'scenario': scenario,
        'model': get_model_name(scenario),
        'pathway': get_pathway_label(scenario),
        'forcing family': get_pathway_family(scenario),
    }
    for scenario in selected_scenarios
]).sort_values(['forcing family', 'pathway']).reset_index(drop=True)

scenario_metadata

### Preview the CH4 and N2O parameter values for the years used in the notebook

In [ ]:
anchor_years = [2020, 2040, 2060, 2080, 2100]

parameter_preview = pd.DataFrame([
    {
        'scenario': scenario,
        'year': year,
        'CF_CH4': prospective_method['scenarios'][scenario]['CF_CH4'][str(year)],
        'CF_N2O': prospective_method['scenarios'][scenario]['CF_N2O'][str(year)],
    }
    for scenario in selected_scenarios
    for year in anchor_years
])

parameter_preview

## 3) Build one `EdgeLCIA` object and reuse it across years and scenarios


In [ ]:
def filter_direct_matches(lca, activity):
    df = lca.generate_cf_table(include_unmatched=False).copy()
    mask = pd.Series(True, index=df.index)
    if 'consumer name' in df.columns:
        mask &= df['consumer name'] == activity['name']
    if 'consumer reference product' in df.columns:
        mask &= df['consumer reference product'] == activity.get('reference product')
    if 'consumer location' in df.columns:
        mask &= df['consumer location'] == activity.get('location')
    return df.loc[mask].copy()


def get_scenario_parameter_value(method_data, scenario, year, parameter_name):
    """Return one prospective CF parameter directly from the method JSON.

    The tomato foreground does not have a direct methane emission, so looking for a
    matched direct methane row in the CF table returns `NaN`. In this notebook we
    therefore report the scenario parameter values themselves, which are exactly
    the numbers used to reevaluate the method for each year and scenario.
    """
    return float(method_data['scenarios'][scenario][parameter_name][str(year)])

In [ ]:
prospective_lca = EdgeLCIA(
    demand={tomato_activity: 1},
    filepath=str(prospective_method_path),
)
prospective_lca.lci()
prospective_lca.map_exchanges()

years = sorted(
    year
    for year in (int(y) for y in prospective_lca.parameters[selected_scenarios[0]]['CF_CH4'].keys())
    if 2020 <= year <= 2100 and year % 20 == 0
)

pd.DataFrame([
    {
        'first year used': min(years),
        'last year used': max(years),
        'year step': 20,
        'number of reevaluations per scenario': len(years),
        'number of MESSAGE scenarios': len(selected_scenarios),
        'total scenario-year combinations': len(selected_scenarios) * len(years),
    }
])

### Evaluate one baseline case to inspect the matched direct CFs in the tomato foreground

In [ ]:
baseline_scenario = selected_scenarios[0]
baseline_year = '2020'

prospective_lca.evaluate_cfs(scenario=baseline_scenario, scenario_idx=baseline_year)
prospective_lca.lcia()

baseline_matches = filter_direct_matches(prospective_lca, tomato_activity)
baseline_matches[[column for column in ['supplier name', 'amount', 'CF', 'impact'] if column in baseline_matches.columns]]


## 4) Iterate through the four representative MESSAGE scenarios every 20 years and track the score

In [ ]:
def build_results_dataframe(lca, method_data, scenarios, years):
    rows = []
    for scenario in scenarios:
        print(scenario)
        for year in years:
            print(year)
            lca.evaluate_cfs(scenario=scenario, scenario_idx=str(year))
            lca.lcia()
            rows.append(
                {
                    'Scenario': scenario,
                    'Model': get_model_name(scenario),
                    'Pathway': get_pathway_label(scenario),
                    'Forcing family': get_pathway_family(scenario),
                    'Year': year,
                    'Methane CF': get_scenario_parameter_value(method_data, scenario, year, 'CF_CH4'),
                    'N2O CF': get_scenario_parameter_value(method_data, scenario, year, 'CF_N2O'),
                    'Score': float(lca.score),
                }
            )
    return pd.DataFrame(rows)

In [ ]:
print(f"{len(selected_scenarios)} scenarios to compute")
results = build_results_dataframe(
    prospective_lca,
    prospective_method,
    selected_scenarios,
    years,
)

results.head(8)

### Compare score snapshots at anchor years


In [ ]:
score_snapshots = (
    results.loc[results['Year'].isin(anchor_years), ['Scenario', 'Year', 'Score']]
    .pivot(index='Scenario', columns='Year', values='Score')
    .loc[selected_scenarios]
)

score_snapshots


### Inspect how the CH4 and N2O CFs move inside the same inventory


In [ ]:
cf_snapshots = (
    results.loc[results['Year'].isin(anchor_years), ['Scenario', 'Year', 'Methane CF', 'N2O CF']]
    .set_index(['Scenario', 'Year'])
    .loc[selected_scenarios]
)

cf_snapshots


### Plot the tomato scores, methane CFs, and N2O CFs side by side

In [ ]:
scenario_colors = {
    scenario: color
    for scenario, color in zip(selected_scenarios, plt.cm.tab20.colors)
}

family_styles = {
    'high forcing': '--',
    'medium forcing': '-.',
    'low forcing': '-',
    'very low forcing': (0, (1, 1)),
    'other': ':',
}

In [ ]:
scenario_labels = {
    scenario: f"{get_pathway_label(scenario)} ({get_pathway_family(scenario)})"
    for scenario in selected_scenarios
}

fig, (ax_ch4, ax_n2o, ax_score) = plt.subplots(1, 3, figsize=(18, 4.8), sharex=True)

for scenario in selected_scenarios:
    data = results.loc[results['Scenario'] == scenario].sort_values('Year')
    color = scenario_colors[scenario]
    style = family_styles.get(get_pathway_family(scenario), ':')
    label = scenario_labels[scenario]

    ax_ch4.plot(
        data['Year'],
        data['Methane CF'],
        color=color,
        linestyle=style,
        linewidth=2.2,
        marker='o',
        markersize=4.5,
        label=label,
    )
    ax_n2o.plot(
        data['Year'],
        data['N2O CF'],
        color=color,
        linestyle=style,
        linewidth=2.2,
        marker='o',
        markersize=4.5,
        label=label,
    )
    ax_score.plot(
        data['Year'],
        data['Score'],
        color=color,
        linestyle=style,
        linewidth=2.2,
        marker='o',
        markersize=4.5,
        label=label,
    )

ax_ch4.set_title('Prospective CH4 CF', fontsize=14)
ax_ch4.set_xlabel('Year', fontsize=13)
ax_ch4.set_ylabel('Methane CF [kg CO2-eq / kg CH4]', fontsize=13)
ax_ch4.grid(True, linestyle='--', alpha=0.35)
ax_ch4.tick_params(axis='both', labelsize=12)

ax_n2o.set_title('Prospective N2O CF', fontsize=14)
ax_n2o.set_xlabel('Year', fontsize=13)
ax_n2o.set_ylabel('N2O CF [kg CO2-eq / kg N2O]', fontsize=13)
ax_n2o.grid(True, linestyle='--', alpha=0.35)
ax_n2o.tick_params(axis='both', labelsize=12)

ax_score.set_title('Prospective GWP100 score', fontsize=14)
ax_score.set_xlabel('Year', fontsize=13)
ax_score.set_ylabel('GWP100 score [kg CO2-eq / kg tomatoes]', fontsize=13)
ax_score.grid(True, linestyle='--', alpha=0.35)
ax_score.tick_params(axis='both', labelsize=12)

handles, labels = ax_ch4.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title='Selected MESSAGE pathways',
    loc='lower center',
    ncol=2,
    fontsize=10,
    title_fontsize=11,
    bbox_to_anchor=(0.5, -0.06),
)

plt.tight_layout(rect=(0, 0.13, 1, 1))
plt.show()

## Checkpoint 1

Use `results` to identify which selected scenario gives the lowest 2100 score and which gives the highest 2100 score. Then compare each of them to its own 2020 score.


In [ ]:
# TODO
# Start from `results`, keep only 2020 and 2100, then compute the absolute and relative score change for each scenario.


In [ ]:
comparison_2020_2100 = (
    results.loc[results['Year'].isin([2020, 2100]), ['Scenario', 'Year', 'Score']]
    .pivot(index='Scenario', columns='Year', values='Score')
    .loc[selected_scenarios]
)

comparison_2020_2100['absolute change'] = comparison_2020_2100[2100] - comparison_2020_2100[2020]
comparison_2020_2100['relative change [%]'] = 100 * comparison_2020_2100['absolute change'] / comparison_2020_2100[2020]

comparison_2020_2100.sort_values(2100)


## Recap

After this notebook, you should now be able to:

- load the packaged prospective GWP100 method directly from `edges`
- inspect how CH4 and N2O CFs are stored by scenario and year inside the JSON method
- map one tomato inventory once and reevaluate its CFs across multiple years and representative MESSAGE scenarios with `evaluate_cfs(scenario=..., scenario_idx=...)`
- reduce runtime by evaluating a coarser year grid while preserving the scenario logic
- show that the same tomato dataset can produce different climate scores when only the scenario-specific CF tables change
- prepare for `D3-04`, where the focus shifts from prospective CFs to tracing intermediate product flows in the inventory
